# 05 — Cải thiện model: 7 thực nghiệm có đối chứng

**Mục tiêu:** xác định model có thực sự tốt lên hay không, chứ không phải tăng Accuracy bằng cách dời ngưỡng.

**Nguyên tắc giữ cố định xuyên suốt:** bộ dữ liệu đã làm sạch (31 feature), one-hot `proto`/`service`/`state`,
cách chia train-fit / validation / test-dev / test-final, định nghĩa nhãn, seed 42.
Mỗi thực nghiệm chỉ đổi **một** yếu tố.

**Kỷ luật dữ liệu:** `test-final` chỉ được chạm **một lần duy nhất** ở bước đánh giá cuối.
Mọi lựa chọn model và mọi lựa chọn ngưỡng đều làm trên `validation` hoặc `test-dev`.

| Thực nghiệm | Câu hỏi |
|---|---|
| 0 | Có tái lập được kết quả đã báo cáo không? |
| 1 | Chênh lệch lần 2 vs lần 3 có ý nghĩa thống kê không? |
| 2 | Bỏ 8 feature ô nhiễm làm mất bao nhiêu hiệu năng? |
| 5 | Lệch phân bố train/test có thật không, ở feature nào? |
| 6 | Fuzzers có thật sự chiếm phần lớn False Negative không? |
| 4 | Random Forest có đang là nút thắt không? |
| 3 | Học theo lớp tấn công cụ thể có giúp phát hiện tấn công tốt hơn không? |
| 7 | Ngưỡng vận hành dưới ràng buộc FPR ≤ 10% cho kết quả gì? |

In [1]:
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/5G_IDS_Project'
except ImportError:
    BASE = os.path.abspath('..')
print('Thư mục project:', BASE)

Thư mục project: G:\.shortcut-targets-by-id\1C2Fd9KvthwrCJvr4mxV0l6w7K1lsUbNE\5G_IDS_Project


In [2]:
import json, time
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.inspection import permutation_importance
from sklearn.metrics import (average_precision_score, roc_auc_score, confusion_matrix,
                             classification_report)
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                     cross_val_predict, cross_val_score)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from scipy.stats import ks_2samp

pd.set_option('display.width', 240)

RS = 42
N_TREES = 300
FPR_MAX = 0.10
TARGET_COLS = ['label', 'attack_cat']
CATEGORICAL = ['proto', 'service', 'state']

CONTAMINANT_8 = ['ct_state_ttl', 'sttl', 'dttl', 'ct_dst_ltm', 'ct_srv_dst',
                 'ct_dst_src_ltm', 'ct_srv_src', 'ct_dst_sport_ltm']
CONTAM_ALL = CONTAMINANT_8 + ['ct_srv_ltm']      # ct_srv_ltm không tồn tại trong CSV
USELESS_3 = ['is_sm_ips_ports', 'ct_ftp_cmd', 'is_ftp_login']

cleaned_dir = os.path.join(BASE, 'data', 'cleaned')
models_dir = os.path.join(BASE, 'models')
raw_dir = os.path.join(BASE, 'data', 'raw', 'CSV Files', 'Training and Testing Sets')
out_dir = os.path.join(BASE, 'reports', 'exp05')
os.makedirs(out_dir, exist_ok=True)


def features(df):
    return df.drop(columns=TARGET_COLS).astype({c: str for c in CATEGORICAL})


def make_rf(n_estimators=N_TREES, **rf_params):
    pre = ColumnTransformer([('cat', OneHotEncoder(handle_unknown='ignore'), CATEGORICAL)],
                            remainder='passthrough', verbose_feature_names_out=False)
    params = dict(n_estimators=n_estimators, random_state=RS, n_jobs=-1) | rf_params
    return Pipeline([('preprocess', pre), ('rf', RandomForestClassifier(**params))])


def metrics_at(y_true, proba, threshold):
    """Attack nếu P(Attack) >= threshold. PR-AUC/ROC-AUC không phụ thuộc ngưỡng."""
    y_true = np.asarray(y_true)
    pred = (proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    return {'threshold': float(threshold), 'accuracy': (tp + tn) / len(pred),
            'precision': tp / max(tp + fp, 1), 'recall': tp / max(tp + fn, 1),
            'f1': 2 * tp / max(2 * tp + fp + fn, 1), 'fpr': fp / max(fp + tn, 1),
            'roc_auc': roc_auc_score(y_true, proba), 'pr_auc': average_precision_score(y_true, proba),
            'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp)}


def threshold_at_fpr(y_true, proba, fpr_max=FPR_MAX):
    """Recall cao nhất trong số các ngưỡng thoả FPR <= fpr_max. Chỉ chạy trên dữ liệu chọn model."""
    y_true = np.asarray(y_true)
    neg, pos = y_true == 0, y_true == 1
    best = (None, -1.0)
    for t in np.r_[np.unique(proba), 1.0 + 1e-9]:
        pred = proba >= t
        if (pred & neg).sum() / neg.sum() <= fpr_max:
            rec = (pred & pos).sum() / pos.sum()
            if rec > best[1]:
                best = (float(t), float(rec))
    return best[0]

## Nạp dữ liệu và tái lập đúng cách chia của notebook 03 / 04

`validation` lấy từ train (80/20, stratify theo label, seed 42) — y hệt notebook 03.
`test-dev` / `test-final` đọc từ `data/cleaned/test_split_v3.csv` đã lưu ở notebook 04,
nên cách chia không phụ thuộc phiên bản thư viện.

In [3]:
train = pd.read_parquet(os.path.join(cleaned_dir, 'UNSW_NB15_training-set.parquet'))
test = pd.read_parquet(os.path.join(cleaned_dir, 'UNSW_NB15_testing-set.parquet'))
X, y = features(train), train['label']

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, stratify=y, random_state=RS)

split = pd.read_csv(os.path.join(cleaned_dir, 'test_split_v3.csv'))
dev_i = split.loc[split['split'] == 'test_dev', 'test_index'].values
fin_i = split.loc[split['split'] == 'test_final', 'test_index'].values
test_dev, test_final = test.loc[dev_i], test.loc[fin_i]
X_dev, y_dev = features(test_dev), test_dev['label']
X_fin, y_fin = features(test_final), test_final['label']

sizes = pd.DataFrame({
    'n': [len(X_tr), len(X_val), len(test_dev), len(test_final)],
    'tỷ lệ Attack': [y_tr.mean(), y_val.mean(), y_dev.mean(), y_fin.mean()],
}, index=['train-fit', 'validation', 'test-dev', 'test-final'])
display(sizes.round(4))
print('Số feature trước one-hot:', X.shape[1])

best_params = json.load(open(os.path.join(models_dir, 'random_forest_v2_metrics.json'),
                             encoding='utf-8'))['best_params']
print('Tham số Random Forest lần 2 (dùng lại cho mọi thực nghiệm RF):', best_params)

,n,tỷ lệ Attack
train-fit,79147,0.4964
validation,19787,0.4964
test-dev,25283,0.3665
test-final,25284,0.3665


Số feature trước one-hot: 31
Tham số Random Forest lần 2 (dùng lại cho mọi thực nghiệm RF): {'min_samples_leaf': 1, 'max_samples': 0.7, 'max_features': 0.5, 'max_depth': 20, 'class_weight': None}


## Thực nghiệm 0 — Tái lập baseline

Nạp đúng ba model đã lưu và đo lại trên `test-final`. Nếu không khớp số đã báo cáo thì
mọi phân tích phía sau đều vô nghĩa.

In [4]:
run1 = joblib.load(os.path.join(models_dir, 'random_forest.joblib'))
run2 = joblib.load(os.path.join(models_dir, 'random_forest_v2.joblib'))
run3_art = joblib.load(os.path.join(models_dir, 'random_forest_v3_calibrated.joblib'))
run3 = run3_art['model']

v3m = json.load(open(os.path.join(models_dir, 'random_forest_v3_metrics.json'), encoding='utf-8'))
thr1, thr2, thr3 = 0.5, float(run2.best_threshold_), float(v3m['threshold_C_dev'])

proba_runs = {
    'run1': run1.predict_proba(X_fin)[:, 1],
    'run2': run2.estimator_.predict_proba(X_fin)[:, 1],
    'run3': run3.predict_proba(X_fin)[:, 1],
}
repro = {
    'run1 @0.500': metrics_at(y_fin, proba_runs['run1'], thr1),
    'run2 @0.515': metrics_at(y_fin, proba_runs['run2'], thr2),
    'run3 @0.711': metrics_at(y_fin, proba_runs['run3'], thr3),
}
cols = ['accuracy', 'precision', 'recall', 'f1', 'fpr', 'roc_auc', 'pr_auc', 'TN', 'FP', 'FN', 'TP']
display(pd.DataFrame(repro).T[cols].round(4))

reported = {
    'run1 @0.500': dict(accuracy=.8156, precision=.6773, recall=.9491, f1=.7905, fpr=.2616, pr_auc=.9264),
    'run2 @0.515': dict(accuracy=.8287, precision=.6965, recall=.9438, f1=.8015, fpr=.2379, pr_auc=.9420),
    'run3 @0.711': dict(accuracy=.8820, precision=.8251, recall=.8605, f1=.8424, fpr=.1055, pr_auc=.9416),
}
for k, exp in reported.items():
    diffs = {m: abs(repro[k][m] - v) for m, v in exp.items()}
    print(f'{k}: sai lệch lớn nhất so với số đã báo cáo = {max(diffs.values()):.5f} ({max(diffs, key=diffs.get)})')

,accuracy,precision,recall,f1,fpr,roc_auc,pr_auc,TN,FP,FN,TP
run1 @0.500,0.8156,0.6773,0.9491,0.7905,0.2616,0.9548,0.9264,11827.0,4190.0,472.0,8795.0
run2 @0.515,0.8287,0.6965,0.9438,0.8015,0.2379,0.9600,0.9420,12206.0,3811.0,521.0,8746.0
run3 @0.711,0.8820,0.8251,0.8605,0.8424,0.1055,0.9595,0.9416,14327.0,1690.0,1293.0,7974.0


run1 @0.500: sai lệch lớn nhất so với số đã báo cáo = 0.00003 (recall)
run2 @0.515: sai lệch lớn nhất so với số đã báo cáo = 0.00005 (pr_auc)
run3 @0.711: sai lệch lớn nhất so với số đã báo cáo = 0.00003 (f1)


### Baseline trong-giao-thức (RF-A)

Ba model đã lưu được fit trên những lượng dữ liệu khác nhau (lần 2 fit trên toàn bộ train,
lần 3 fit trên 80%). Để mọi thực nghiệm mới so sánh được với nhau, ta dựng **RF-A**:
đúng tham số lần 2, fit trên `train-fit` (79.147 dòng) — tức là model A của notebook 04.
Từ đây về sau, RF-A là mốc so sánh duy nhất.

In [5]:
t0 = time.time()
rf_A = make_rf(**best_params).fit(X_tr, y_tr)
print(f'RF-A fit xong trong {time.time() - t0:.0f}s')

pA = {'validation': rf_A.predict_proba(X_val)[:, 1],
      'test-dev': rf_A.predict_proba(X_dev)[:, 1],
      'test-final': rf_A.predict_proba(X_fin)[:, 1]}
ys = {'validation': y_val, 'test-dev': y_dev, 'test-final': y_fin}
display(pd.DataFrame({f'RF-A | {s}': metrics_at(ys[s], pA[s], 0.5) for s in ys}).T[cols].round(4))

RF-A fit xong trong 51s


,accuracy,precision,recall,f1,fpr,roc_auc,pr_auc,TN,FP,FN,TP
RF-A | validation,0.9322,0.9104,0.9577,0.9335,0.0929,0.9860,0.9849,9039.0,926.0,415.0,9407.0
RF-A | test-dev,0.8154,0.6767,0.9504,0.7905,0.2627,0.9600,0.9429,11810.0,4207.0,460.0,8806.0
RF-A | test-final,0.8186,0.6813,0.9491,0.7932,0.2569,0.9594,0.9414,11903.0,4114.0,472.0,8795.0


## Thực nghiệm 1 — Bootstrap: chênh lệch nào là thật?

1.000 lần lấy mẫu có hoàn lại trên `test-final`. Vì mọi model dùng **cùng các dòng**,
ta lấy mẫu **theo cặp** (paired bootstrap) rồi lấy phân phối của *hiệu*.
So hai khoảng tin cậy rời nhau là sai — hiệu số mới là đại lượng cần kiểm tra.

Hàng `run3 @0.500` là model lần 3 nhưng cắt ở đúng điểm vận hành của lần 2:
nó tách phần "đổi model" ra khỏi phần "đổi ngưỡng".

In [6]:
BOOT_METRICS = ['pr_auc', 'roc_auc', 'accuracy', 'precision', 'recall', 'f1', 'fpr']


def _point(y_true, p, thr):
    pred = p >= thr
    pos = y_true == 1
    tp = np.count_nonzero(pred & pos); fp = np.count_nonzero(pred & ~pos)
    fn = np.count_nonzero(~pred & pos); tn = np.count_nonzero(~pred & ~pos)
    return {'pr_auc': average_precision_score(y_true, p), 'roc_auc': roc_auc_score(y_true, p),
            'accuracy': (tp + tn) / len(y_true), 'precision': tp / max(tp + fp, 1),
            'recall': tp / max(tp + fn, 1), 'f1': 2 * tp / max(2 * tp + fp + fn, 1),
            'fpr': fp / max(fp + tn, 1)}


def paired_bootstrap(y_true, models, B=1000, seed=RS, alpha=0.05):
    y_true = np.asarray(y_true); n = len(y_true)
    rng = np.random.default_rng(seed)
    point = {k: _point(y_true, np.asarray(v[0]), v[1]) for k, v in models.items()}
    draws = {k: np.full((B, len(BOOT_METRICS)), np.nan) for k in models}
    for b in range(B):
        idx = rng.integers(0, n, n)
        yb = y_true[idx]
        if yb.min() == yb.max():
            continue
        for k, (p, thr) in models.items():
            r = _point(yb, np.asarray(p)[idx], thr)
            draws[k][b] = [r[m] for m in BOOT_METRICS]
    lo, hi = 100 * alpha / 2, 100 * (1 - alpha / 2)
    ci = {k: {m: (np.nanpercentile(d[:, i], lo), np.nanpercentile(d[:, i], hi))
              for i, m in enumerate(BOOT_METRICS)} for k, d in draws.items()}
    return pd.DataFrame(point).T[BOOT_METRICS], ci, draws


def diff_ci(draws, a, b, alpha=0.05):
    d = draws[a] - draws[b]
    lo, hi = 100 * alpha / 2, 100 * (1 - alpha / 2)
    out = {}
    for i, m in enumerate(BOOT_METRICS):
        col = d[:, i][~np.isnan(d[:, i])]
        l, h = np.percentile(col, lo), np.percentile(col, hi)
        out[m] = {'hiệu_TB': float(col.mean()), 'CI_thấp': float(l), 'CI_cao': float(h),
                  'loại_trừ_0': bool(l > 0 or h < 0),
                  'p_hai_phía': float(2 * min((col <= 0).mean(), (col >= 0).mean()))}
    return pd.DataFrame(out).T

In [7]:
models_e1 = {
    'run1 @0.500': (proba_runs['run1'], thr1),
    'run2 @0.515': (proba_runs['run2'], thr2),
    'run3 @0.711': (proba_runs['run3'], thr3),
    'run3 @0.500': (proba_runs['run3'], 0.5),   # cùng model lần 3, điểm vận hành của lần 2
}
pt_e1, ci_e1, draws_e1 = paired_bootstrap(y_fin, models_e1, B=1000, seed=RS)
print('Ước lượng điểm trên test-final:')
display(pt_e1.round(4))
print('Khoảng tin cậy 95% (1.000 lần lấy mẫu):')
display(pd.DataFrame({k: {m: f'[{v[m][0]:.4f}, {v[m][1]:.4f}]' for m in BOOT_METRICS}
                      for k, v in ci_e1.items()}).T)

for a, b in [('run2 @0.515', 'run1 @0.500'), ('run3 @0.711', 'run2 @0.515'), ('run3 @0.500', 'run2 @0.515')]:
    print(f'\n--- hiệu: {a}  −  {b} ---')
    display(diff_ci(draws_e1, a, b).round(5))

Ước lượng điểm trên test-final:


,pr_auc,roc_auc,accuracy,precision,recall,f1,fpr
run1 @0.500,0.9264,0.9548,0.8156,0.6773,0.9491,0.7905,0.2616
run2 @0.515,0.9420,0.9600,0.8287,0.6965,0.9438,0.8015,0.2379
run3 @0.711,0.9416,0.9595,0.8820,0.8251,0.8605,0.8424,0.1055
run3 @0.500,0.9416,0.9595,0.8289,0.6976,0.9413,0.8013,0.2361


Khoảng tin cậy 95% (1.000 lần lấy mẫu):


,pr_auc,roc_auc,accuracy,precision,recall,f1,fpr
run1 @0.500,"[0.9220, 0.9304]","[0.9526, 0.9569]","[0.8110, 0.8204]","[0.6695, 0.6857]","[0.9449, 0.9534]","[0.7848, 0.7964]","[0.2546, 0.2682]"
run2 @0.515,"[0.9386, 0.9450]","[0.9580, 0.9620]","[0.8244, 0.8336]","[0.6886, 0.7048]","[0.9387, 0.9483]","[0.7958, 0.8074]","[0.2311, 0.2441]"
run3 @0.711,"[0.9383, 0.9445]","[0.9575, 0.9614]","[0.8781, 0.8860]","[0.8174, 0.8328]","[0.8534, 0.8676]","[0.8366, 0.8479]","[0.1007, 0.1100]"
run3 @0.500,"[0.9383, 0.9445]","[0.9575, 0.9614]","[0.8244, 0.8336]","[0.6898, 0.7055]","[0.9364, 0.9459]","[0.7955, 0.8070]","[0.2295, 0.2425]"



--- hiệu: run2 @0.515  −  run1 @0.500 ---


,hiệu_TB,CI_thấp,CI_cao,loại_trừ_0,p_hai_phía
pr_auc,0.015578,0.013454,0.017893,True,0.0
roc_auc,0.005212,0.004425,0.006002,True,0.0
accuracy,0.013033,0.010402,0.015662,True,0.0
precision,0.019175,0.015831,0.022449,True,0.0
recall,-0.005322,-0.008568,-0.001972,True,0.002
f1,0.010993,0.008301,0.013667,True,0.0
fpr,-0.023646,-0.027338,-0.019913,True,0.0



--- hiệu: run3 @0.711  −  run2 @0.515 ---


,hiệu_TB,CI_thấp,CI_cao,loại_trừ_0,p_hai_phía
pr_auc,-0.000346,-0.000984,0.000277,False,0.272
roc_auc,-0.000494,-0.000889,-0.000106,True,0.01
accuracy,0.053286,0.049241,0.057349,True,0.0
precision,0.12856,0.122822,0.134673,True,0.0
recall,-0.083342,-0.089064,-0.077883,True,0.0
f1,0.040862,0.03602,0.045822,True,0.0
fpr,-0.132293,-0.137794,-0.126923,True,0.0



--- hiệu: run3 @0.500  −  run2 @0.515 ---


,hiệu_TB,CI_thấp,CI_cao,loại_trừ_0,p_hai_phía
pr_auc,-0.000346,-0.000984,0.000277,False,0.272
roc_auc,-0.000494,-0.000889,-0.000106,True,0.01
accuracy,0.000227,-0.001622,0.002058,False,0.844
precision,0.001039,-0.001364,0.003472,False,0.43
recall,-0.002469,-0.004737,-0.000321,True,0.032
f1,-0.000206,-0.002177,0.0017,False,0.86
fpr,-0.001785,-0.004457,0.000933,False,0.194


## Thực nghiệm 2 — Ablation 8 feature ô nhiễm (31 vs 39)

Để hai nhánh **chỉ** khác nhau ở 8 cột đó, tập dòng sống sót phải được quyết định
trong không gian 31 feature **đúng như notebook 01**, rồi mới đính 8 cột kia vào.
Ô kiểm chứng bên dưới khẳng định nhánh A **trùng khít** bộ cleaned chính thức —
nếu không trùng thì ablation không còn là đối chứng.

⚠️ Nhánh B chỉ phục vụ mục đích khoa học. **Không** ghi đè `data/cleaned`, **không**
dùng làm model chính thức.

In [8]:
from fastai.tabular.all import df_shrink


def clean_keep_contaminants(csv_path):
    df = pd.read_csv(csv_path, skipinitialspace=True, low_memory=False, encoding='utf-8')
    df.drop(columns=['id'], inplace=True)
    df = df_shrink(df, skip=[], obj2cat=True, int2uint=False)
    df.drop(columns=[c for c in USELESS_3 if c in df.columns], inplace=True)
    contam = [c for c in CONTAM_ALL if c in df.columns]
    feat31 = [c for c in df.columns if c not in TARGET_COLS + contam]
    df = df.drop_duplicates(subset=feat31 + TARGET_COLS)          # như notebook 01
    n_lab = df.groupby(feat31, observed=True, dropna=False)['label'].transform('nunique')
    df = df[n_lab == 1].reset_index(drop=True)
    return df, feat31, contam


arms_data = {}
for split_name, official in [('training', train), ('testing', test)]:
    df39, feat31, contam = clean_keep_contaminants(os.path.join(raw_dir, f'UNSW_NB15_{split_name}-set.csv'))
    rebuilt = df39[list(official.columns)].reset_index(drop=True)
    identical = rebuilt.equals(official.reset_index(drop=True))
    print(f'[{split_name}] dựng lại {df39.shape} | nhánh 31-feature {rebuilt.shape} '
          f'| chính thức {official.shape} | trùng khít = {identical}')
    arms_data[split_name] = df39

assert all(arms_data[s][list(o.columns)].reset_index(drop=True).equals(o.reset_index(drop=True))
           for s, o in [('training', train), ('testing', test)]), 'Ablation không phải đối chứng!'
print('\nNhánh A == bộ cleaned chính thức. Ablation hợp lệ.')
print('8 cột khôi phục ở nhánh B:', contam)

C:\venvs\ids5g\Lib\site-packages\torch\jit\_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


[training] dựng lại (98934, 41) | nhánh 31-feature (98934, 33) | chính thức (98934, 33) | trùng khít = True


[testing] dựng lại (50567, 41) | nhánh 31-feature (50567, 33) | chính thức (50567, 33) | trùng khít = True

Nhánh A == bộ cleaned chính thức. Ablation hợp lệ.
8 cột khôi phục ở nhánh B: ['ct_state_ttl', 'sttl', 'dttl', 'ct_dst_ltm', 'ct_srv_dst', 'ct_dst_src_ltm', 'ct_srv_src', 'ct_dst_sport_ltm']


In [9]:
tr39, te39 = arms_data['training'], arms_data['testing']
feat39 = feat31 + contam


def feats(df, cols):
    return df[cols].astype({c: str for c in CATEGORICAL})


idx_tr, idx_val = train_test_split(tr39.index, test_size=0.2, stratify=tr39['label'], random_state=RS)

abl_results, abl_proba, abl_models = {}, {}, {}
for arm, cols_arm in [('A: 31 feature (chính thức)', feat31), ('B: 39 feature (+8 ô nhiễm)', feat39)]:
    Xa = feats(tr39, cols_arm)
    m = make_rf(**best_params).fit(Xa.loc[idx_tr], tr39.loc[idx_tr, 'label'])
    width = m.named_steps['preprocess'].transform(Xa.iloc[:5]).shape[1]
    print(f'{arm}: {len(cols_arm)} feature -> {width} cột sau one-hot')
    for sname, Xs, ysub in [('validation', Xa.loc[idx_val], tr39.loc[idx_val, 'label']),
                            ('test-dev', feats(te39.loc[dev_i], cols_arm), te39.loc[dev_i, 'label']),
                            ('test-final', feats(te39.loc[fin_i], cols_arm), te39.loc[fin_i, 'label'])]:
        p = m.predict_proba(Xs)[:, 1]
        abl_results[f'{arm} | {sname}'] = metrics_at(ysub, p, 0.5)
        abl_proba[f'{arm}|{sname}'] = p
    abl_models[arm] = (m, Xa.loc[idx_val], tr39.loc[idx_val, 'label'])

print('\nNgưỡng cố định 0,50 ở cả hai nhánh:')
display(pd.DataFrame(abl_results).T[cols].round(4))

_, _, draws_abl = paired_bootstrap(y_fin, {
    'B39': (abl_proba['B: 39 feature (+8 ô nhiễm)|test-final'], 0.5),
    'A31': (abl_proba['A: 31 feature (chính thức)|test-final'], 0.5)}, B=1000, seed=RS)
print('Hiệu B(39) − A(31) trên test-final:')
display(diff_ci(draws_abl, 'B39', 'A31').round(5))

A: 31 feature (chính thức): 31 feature -> 183 cột sau one-hot


B: 39 feature (+8 ô nhiễm): 39 feature -> 191 cột sau one-hot



Ngưỡng cố định 0,50 ở cả hai nhánh:


,accuracy,precision,recall,f1,fpr,roc_auc,pr_auc,TN,FP,FN,TP
A: 31 feature (chính thức) | validation,0.9322,0.9104,0.9577,0.9335,0.0929,0.9860,0.9849,9039.0,926.0,415.0,9407.0
A: 31 feature (chính thức) | test-dev,0.8154,0.6767,0.9504,0.7905,0.2627,0.9600,0.9429,11810.0,4207.0,460.0,8806.0
A: 31 feature (chính thức) | test-final,0.8186,0.6813,0.9491,0.7932,0.2569,0.9594,0.9414,11903.0,4114.0,472.0,8795.0
B: 39 feature (+8 ô nhiễm) | validation,0.9404,0.9260,0.9563,0.9409,0.0754,0.9889,0.9881,9214.0,751.0,429.0,9393.0
B: 39 feature (+8 ô nhiễm) | test-dev,0.8353,0.6974,0.9725,0.8123,0.2441,0.9689,0.9525,12108.0,3909.0,255.0,9011.0
B: 39 feature (+8 ô nhiễm) | test-final,0.8345,0.6969,0.9708,0.8113,0.2443,0.9690,0.9527,12104.0,3913.0,271.0,8996.0


Hiệu B(39) − A(31) trên test-final:


,hiệu_TB,CI_thấp,CI_cao,loại_trừ_0,p_hai_phía
pr_auc,0.011322,0.009256,0.01343,True,0.0
roc_auc,0.009614,0.008249,0.010974,True,0.0
accuracy,0.015908,0.012774,0.019302,True,0.0
precision,0.015579,0.011518,0.019901,True,0.0
recall,0.021724,0.017531,0.026119,True,0.0
f1,0.018146,0.014811,0.02161,True,0.0
fpr,-0.012544,-0.017308,-0.00798,True,0.0


In [10]:
perm_tbl = {}
for arm, (m, Xv, yv) in abl_models.items():
    pi = permutation_importance(m, Xv, yv, scoring='average_precision', n_repeats=5,
                                random_state=RS, n_jobs=1)
    perm_tbl[arm] = pd.Series(pi.importances_mean, index=Xv.columns).sort_values(ascending=False)
    print(f'\n--- permutation importance, top 10 — {arm} ---')
    display(perm_tbl[arm].head(10).round(5))


--- permutation importance, top 10 — A: 31 feature (chính thức) ---


ackdat     0.05010
sbytes     0.02571
dload      0.02015
dbytes     0.01254
smean      0.01136
service    0.01102
synack     0.00835
tcprtt     0.00725
dmean      0.00629
dloss      0.00557
dtype: float64


--- permutation importance, top 10 — B: 39 feature (+8 ô nhiễm) ---


sttl              0.17980
ct_srv_dst        0.00710
dbytes            0.00708
ct_dst_src_ltm    0.00647
ct_state_ttl      0.00628
sbytes            0.00605
dmean             0.00423
smean             0.00378
proto             0.00309
sloss             0.00306
dtype: float64

## Thực nghiệm 5 — Lệch phân bố train/test

Bốn phép đo độc lập: tỷ lệ lớp, thống kê KS theo từng feature số, mức lệch của
biến phân loại, và adversarial validation (train vs test). Cuối cùng đo khoảng chênh
giữa CV trên train và kết quả trên test.

In [11]:
X_test_all = pd.concat([X_dev, X_fin])
y_test_all = pd.concat([y_dev, y_fin])

num_cols = [c for c in X.columns if c not in CATEGORICAL]
ks_rows = []
for c in num_cols:
    a = X[c].to_numpy(dtype=float); b = X_test_all[c].to_numpy(dtype=float)
    ks_rows.append({'feature': c,
                    'KS_tất_cả': ks_2samp(a, b).statistic,
                    'KS_chỉ_Normal': ks_2samp(a[(y == 0).to_numpy()],
                                              b[(y_test_all == 0).to_numpy()]).statistic,
                    'trung_vị_train': float(np.median(a)), 'trung_vị_test': float(np.median(b))})
ks = pd.DataFrame(ks_rows).set_index('feature').sort_values('KS_tất_cả', ascending=False)
print(f'Thống kê KS train vs test, {len(ks)} feature số — top 15:')
display(ks.head(15).round(4))

watch = [c for c in ['ackdat', 'synack', 'tcprtt', 'sbytes', 'dur', 'sinpkt', 'sload'] if c in ks.index]
print('Các feature review nêu tên, kèm thứ hạng lệch:')
display(ks.loc[watch].round(4).assign(hạng=[int(ks.index.get_loc(c)) + 1 for c in watch]))

Thống kê KS train vs test, 28 feature số — top 15:


,KS_tất_cả,KS_chỉ_Normal,trung_vị_train,trung_vị_test
feature,,,,
ackdat,0.0984,0.2698,0.0002,0.0288
dbytes,0.0806,0.1904,678.0000,354.0000
dmean,0.0779,0.2061,80.0000,66.0000
dloss,0.0669,0.1669,2.0000,1.0000
dload,0.0654,0.2391,9923.3599,6970.7983
sbytes,0.0639,0.1316,1012.0000,978.0000
spkts,0.0635,0.1374,10.0000,10.0000
dpkts,0.0614,0.1604,8.0000,8.0000
tcprtt,0.0603,0.2258,0.0009,0.0530


Các feature review nêu tên, kèm thứ hạng lệch:


,KS_tất_cả,KS_chỉ_Normal,trung_vị_train,trung_vị_test,hạng
feature,,,,,
ackdat,0.0984,0.2698,0.0002,0.0288,1
synack,0.0590,0.2250,0.0007,0.0100,10
tcprtt,0.0603,0.2258,0.0009,0.0530,9
sbytes,0.0639,0.1316,1012.0000,978.0000,6
dur,0.0342,0.1491,0.4190,0.4678,17
sinpkt,0.0468,0.1762,26.7511,34.1037,13
sload,0.0416,0.1546,41340.5742,28992.0547,16


In [12]:
cat_rows = []
for c in CATEGORICAL:
    p = X[c].value_counts(normalize=True); q = X_test_all[c].value_counts(normalize=True)
    allv = p.index.union(q.index)
    cat_rows.append({'feature': c, 'số_mức_train': len(p), 'số_mức_test': len(q),
                     'mức_lạ_với_train': len(set(q.index) - set(p.index)),
                     'TVD': 0.5 * float(np.abs(p.reindex(allv, fill_value=0) - q.reindex(allv, fill_value=0)).sum())})
display(pd.DataFrame(cat_rows).set_index('feature').round(4))

slow = {}
for name, Xs, ysub in [('train', X, y), ('test', X_test_all, y_test_all)]:
    s = Xs['tcprtt'].to_numpy() > 0.01
    atk = (ysub == 1).to_numpy()
    slow[name] = {'% Normal bắt tay chậm': float(s[~atk].mean()),
                  '% Attack bắt tay chậm': float(s[atk].mean()),
                  'P(Attack | chậm)': float(atk[s].mean()),
                  'tỷ lệ Attack:Normal trong vùng chậm': float(atk[s].sum() / max((~atk[s]).sum(), 1))}
print('Quy tắc "bắt tay TCP chậm => Attack" (tcprtt > 0,01 s):')
display(pd.DataFrame(slow).T.round(4))

,số_mức_train,số_mức_test,mức_lạ_với_train,TVD
feature,,,,
proto,133,131,0,0.0220
service,13,13,0,0.0720
state,9,7,2,0.0152


Quy tắc "bắt tay TCP chậm => Attack" (tcprtt > 0,01 s):


,% Normal bắt tay chậm,% Attack bắt tay chậm,P(Attack | chậm),tỷ lệ Attack:Normal trong vùng chậm
train,0.1899,0.7748,0.8009,4.0216
test,0.4128,0.7492,0.5122,1.0501


In [13]:
t0 = time.time()
A_adv = pd.concat([X, X_test_all], ignore_index=True)
b_adv = np.r_[np.zeros(len(X)), np.ones(len(X_test_all))]
adv = cross_val_predict(make_rf(n_estimators=100), A_adv, b_adv,
                        cv=StratifiedKFold(3, shuffle=True, random_state=RS),
                        method='predict_proba')[:, 1]
lab_all = np.r_[y.to_numpy(), y_test_all.to_numpy()]
print(f'Adversarial validation ({time.time() - t0:.0f}s) — 0,50 nghĩa là không phân biệt được:')
print(f'  AUC toàn bộ      : {roc_auc_score(b_adv, adv):.4f}')
print(f'  AUC chỉ Normal   : {roc_auc_score(b_adv[lab_all == 0], adv[lab_all == 0]):.4f}')
print(f'  AUC chỉ Attack   : {roc_auc_score(b_adv[lab_all == 1], adv[lab_all == 1]):.4f}')

Adversarial validation (134s) — 0,50 nghĩa là không phân biệt được:
  AUC toàn bộ      : 0.8012
  AUC chỉ Normal   : 0.8722
  AUC chỉ Attack   : 0.6689


In [14]:
t0 = time.time()
cv_scores = cross_val_score(make_rf(**best_params), X_tr, y_tr, scoring='average_precision',
                            cv=StratifiedKFold(5, shuffle=True, random_state=RS), n_jobs=1)
test_pr = metrics_at(y_fin, pA['test-final'], 0.5)['pr_auc']
print(f'CV 5-fold PR-AUC trên train-fit ({time.time() - t0:.0f}s): {np.round(cv_scores, 4).tolist()}')
print(f'  trung bình {cv_scores.mean():.4f} (std {cv_scores.std():.4f})')
print(f'  PR-AUC trên test-final của cùng cấu hình: {test_pr:.4f}')
print(f'  khoảng chênh: {cv_scores.mean() - test_pr:.4f}')

CV 5-fold PR-AUC trên train-fit (182s): [0.984, 0.9828, 0.9835, 0.9844, 0.9802]
  trung bình 0.9830 (std 0.0015)
  PR-AUC trên test-final của cùng cấu hình: 0.9414
  khoảng chênh: 0.0416


## Thực nghiệm 6 — Fuzzers có thật sự là nguồn False Negative chính?

Lưu ý: với model **nhị phân**, Precision theo từng loại tấn công không định nghĩa được
(mọi False Positive đều là traffic Normal, không mang `attack_cat`). Vì vậy ở đây báo cáo
Recall và số FN theo từng loại; Precision/F1 theo lớp nằm ở thực nghiệm đa lớp.

In [15]:
cat_fin = test_final['attack_cat'].astype(str).to_numpy()
yf = y_fin.to_numpy()


def fn_breakdown(proba, thr):
    pred = (proba >= thr).astype(int)
    atk = yf == 1
    rows = {}
    for c in sorted(set(cat_fin[atk])):
        m = atk & (cat_fin == c)
        hit = pred[m] == 1
        rows[c] = {'n': int(m.sum()), 'TP': int(hit.sum()), 'FN': int((~hit).sum()),
                   'recall': float(hit.mean())}
    t = pd.DataFrame(rows).T.sort_values('FN', ascending=False)
    t['tỷ_trọng_FN'] = t['FN'] / t['FN'].sum()
    return t


for thr in [0.5, 0.711]:
    t = fn_breakdown(pA['test-final'], thr)
    print(f'--- ngưỡng {thr} — tổng FN = {int(t["FN"].sum())} ---')
    display(t.round(4))

--- ngưỡng 0.5 — tổng FN = 472 ---


,n,TP,FN,recall,tỷ_trọng_FN
Fuzzers,2218.0,1827.0,391.0,0.8237,0.8284
Exploits,3780.0,3731.0,49.0,0.9870,0.1038
DoS,839.0,829.0,10.0,0.9881,0.0212
Shellcode,185.0,176.0,9.0,0.9514,0.0191
Generic,715.0,708.0,7.0,0.9902,0.0148
Reconnaissance,1131.0,1126.0,5.0,0.9956,0.0106
Backdoor,164.0,163.0,1.0,0.9939,0.0021
Analysis,213.0,213.0,0.0,1.0000,0.0000
Worms,22.0,22.0,0.0,1.0000,0.0000


--- ngưỡng 0.711 — tổng FN = 1364 ---


,n,TP,FN,recall,tỷ_trọng_FN
Fuzzers,2218.0,1179.0,1039.0,0.5316,0.7617
Exploits,3780.0,3593.0,187.0,0.9505,0.1371
DoS,839.0,794.0,45.0,0.9464,0.0330
Shellcode,185.0,143.0,42.0,0.7730,0.0308
Reconnaissance,1131.0,1106.0,25.0,0.9779,0.0183
Generic,715.0,695.0,20.0,0.9720,0.0147
Analysis,213.0,209.0,4.0,0.9812,0.0029
Backdoor,164.0,163.0,1.0,0.9939,0.0007
Worms,22.0,21.0,1.0,0.9545,0.0007


In [16]:
groups = {'Normal': yf == 0,
          'Fuzzers': (yf == 1) & (cat_fin == 'Fuzzers'),
          'Attack trừ Fuzzers': (yf == 1) & (cat_fin != 'Fuzzers')}
qs = [0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95]
pf = pA['test-final']
tbl = pd.DataFrame({k: np.quantile(pf[m], qs) for k, m in groups.items()},
                   index=[f'q{int(q*100)}' for q in qs])
tbl.loc['n'] = [int(m.sum()) for m in groups.values()]
tbl.loc['trung bình'] = [float(pf[m].mean()) for m in groups.values()]
print('Phân phối P(Attack) theo nhóm (test-final, RF-A):')
display(tbl.round(4))
print('Tỷ lệ có P(Attack) < 0,5:', {k: round(float((pf[m] < 0.5).mean()), 4) for k, m in groups.items()})

rows = []
for c in num_cols:
    v = X_fin[c].to_numpy(dtype=float)
    rows.append({'feature': c,
                 'KS_Fuzzers_vs_Normal': ks_2samp(v[groups['Fuzzers']], v[groups['Normal']]).statistic,
                 'KS_Attack_khác_vs_Normal': ks_2samp(v[groups['Attack trừ Fuzzers']], v[groups['Normal']]).statistic})
ov = pd.DataFrame(rows).set_index('feature')
ov['chênh_lệch_tách_biệt'] = ov['KS_Attack_khác_vs_Normal'] - ov['KS_Fuzzers_vs_Normal']
print(f"KS trung bình so với Normal — Fuzzers {ov['KS_Fuzzers_vs_Normal'].mean():.4f} "
      f"| các tấn công khác {ov['KS_Attack_khác_vs_Normal'].mean():.4f}")
display(ov.sort_values('chênh_lệch_tách_biệt', ascending=False).head(10).round(4))

svc = pd.DataFrame({k: X_fin.loc[m, 'service'].value_counts(normalize=True)
                    for k, m in groups.items()}).fillna(0)
print('Cơ cấu service theo nhóm (top 6):')
display(svc.sort_values('Fuzzers', ascending=False).head(6).round(4))

Phân phối P(Attack) theo nhóm (test-final, RF-A):


,Normal,Fuzzers,Attack trừ Fuzzers
q5,0.0000,0.3400,0.7239
q10,0.0000,0.4173,0.8548
q25,0.0000,0.5559,0.9701
q50,0.0000,0.7378,0.9958
q75,0.5074,0.9200,0.9998
q90,0.7042,0.9933,1.0000
q95,0.8192,1.0000,1.0000
n,16017.0000,2218.0000,7049.0000
trung bình,0.2331,0.7199,0.9546


Tỷ lệ có P(Attack) < 0,5: {'Normal': 0.7431, 'Fuzzers': 0.1763, 'Attack trừ Fuzzers': 0.0115}


KS trung bình so với Normal — Fuzzers 0.2191 | các tấn công khác 0.1766


,KS_Fuzzers_vs_Normal,KS_Attack_khác_vs_Normal,chênh_lệch_tách_biệt
feature,,,
ct_src_ltm,0.1808,0.2660,0.0852
trans_depth,0.0741,0.1203,0.0462
ct_flw_http_mthd,0.0741,0.1202,0.0461
sbytes,0.1641,0.1958,0.0317
smean,0.1366,0.1585,0.0219
dwin,0.0204,0.0342,0.0139
swin,0.0360,0.0497,0.0137
stcpb,0.0237,0.0362,0.0126
dtcpb,0.0236,0.0350,0.0114


Cơ cấu service theo nhóm (top 6):


,Normal,Fuzzers,Attack trừ Fuzzers
service,,,
-,0.7172,0.8855,0.5084
http,0.1129,0.0631,0.2827
ftp,0.0244,0.0464,0.0218
dns,0.0901,0.0050,0.0602
dhcp,0.0000,0.0000,0.0017
ftp-data,0.0292,0.0000,0.0159


## Thực nghiệm 4 — Boosting vs Random Forest

Giữ nguyên feature, cách chia, tiền xử lý, nhãn và giao thức đánh giá; **chỉ đổi thuật toán**.
Không dò tham số diện rộng — một cấu hình hợp lý, có early stopping, và ghi lại tham số đã dùng.
Môi trường không có LightGBM/XGBoost nên chỉ dùng `HistGradientBoostingClassifier`
(không cài thêm phụ thuộc chỉ để chạy một thực nghiệm).

In [17]:
store, summary = {}, {}


def record(name, p_val, p_dev, p_fin, thr=0.5):
    store[f'{name}|val'], store[f'{name}|dev'], store[f'{name}|fin'] = p_val, p_dev, p_fin
    summary[f'{name} | validation'] = metrics_at(y_val, p_val, thr)
    summary[f'{name} | test-dev'] = metrics_at(y_dev, p_dev, thr)
    summary[f'{name} | test-final'] = metrics_at(y_fin, p_fin, thr)


HGB_PARAMS = dict(max_iter=500, learning_rate=0.1, max_leaf_nodes=31, min_samples_leaf=20,
                  l2_regularization=0.0, early_stopping=True, validation_fraction=0.1,
                  n_iter_no_change=20, random_state=RS)
print('Tham số HistGradientBoosting:', HGB_PARAMS)

# cùng một phép mã hoá one-hot, chỉ khác ở chỗ HistGB cần ma trận dày
pre_dense = ColumnTransformer(
    [('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), CATEGORICAL)],
    remainder='passthrough', verbose_feature_names_out=False)
hgb = Pipeline([('preprocess', pre_dense), ('hgb', HistGradientBoostingClassifier(**HGB_PARAMS))])
t0 = time.time()
hgb.fit(X_tr, y_tr)
print(f'  fit {time.time() - t0:.0f}s | số vòng lặp thực dùng: {hgb.named_steps["hgb"].n_iter_}')
record('HGB boosting', hgb.predict_proba(X_val)[:, 1], hgb.predict_proba(X_dev)[:, 1],
       hgb.predict_proba(X_fin)[:, 1])

Tham số HistGradientBoosting: {'max_iter': 500, 'learning_rate': 0.1, 'max_leaf_nodes': 31, 'min_samples_leaf': 20, 'l2_regularization': 0.0, 'early_stopping': True, 'validation_fraction': 0.1, 'n_iter_no_change': 20, 'random_state': 42}


  fit 8s | số vòng lặp thực dùng: 288


## Thực nghiệm 3 — Phân loại đa lớp theo `attack_cat`

`attack_cat` **chỉ** dùng làm nhãn, không bao giờ làm feature đầu vào.
Điểm nhị phân suy ra bằng `P(Attack) = 1 − P(Normal)`, rồi đo trên đúng giao thức cũ.

In [18]:
y_tr_cat = train.loc[X_tr.index, 'attack_cat'].astype(str)
y_fin_cat = test_final['attack_cat'].astype(str)
print('Phân bố lớp khi huấn luyện:')
display(y_tr_cat.value_counts())

t0 = time.time()
mc = make_rf(**best_params).fit(X_tr, y_tr_cat)
print(f'RF đa lớp fit {time.time() - t0:.0f}s | các lớp: {list(mc.classes_)}')
norm_i = list(mc.classes_).index('Normal')


def attack_score(model, Xs):
    return 1.0 - model.predict_proba(Xs)[:, norm_i]


record('RF đa lớp (1−P(Normal))', attack_score(mc, X_val), attack_score(mc, X_dev), attack_score(mc, X_fin))
record('RF-A nhị phân (baseline)', pA['validation'], pA['test-dev'], pA['test-final'])

print('Tất cả các nhánh, ngưỡng cố định 0,50:')
display(pd.DataFrame(summary).T[['pr_auc', 'roc_auc', 'accuracy', 'precision', 'recall', 'f1', 'fpr']].round(4))

Phân bố lớp khi huấn luyện:


attack_cat
Normal            39860
Exploits          15691
Fuzzers           11436
Reconnaissance     5101
DoS                2815
Generic            1227
Backdoor           1053
Analysis           1048
Shellcode           812
Worms               104
Name: count, dtype: int64

RF đa lớp fit 76s | các lớp: ['Analysis', 'Backdoor', 'DoS', 'Exploits', 'Fuzzers', 'Generic', 'Normal', 'Reconnaissance', 'Shellcode', 'Worms']


Tất cả các nhánh, ngưỡng cố định 0,50:


,pr_auc,roc_auc,accuracy,precision,recall,f1,fpr
HGB boosting | validation,0.9835,0.9851,0.9288,0.9095,0.9512,0.9299,0.0933
HGB boosting | test-dev,0.9433,0.9599,0.8214,0.6858,0.9464,0.7953,0.2509
HGB boosting | test-final,0.9417,0.9591,0.8231,0.6878,0.9473,0.7970,0.2487
RF đa lớp (1−P(Normal)) | validation,0.9842,0.9862,0.9334,0.9175,0.9513,0.9341,0.0843
RF đa lớp (1−P(Normal)) | test-dev,0.9453,0.9618,0.8266,0.6937,0.9431,0.7994,0.2409
RF đa lớp (1−P(Normal)) | test-final,0.9427,0.9607,0.8270,0.6945,0.9424,0.7997,0.2398
RF-A nhị phân (baseline) | validation,0.9849,0.9860,0.9322,0.9104,0.9577,0.9335,0.0929
RF-A nhị phân (baseline) | test-dev,0.9429,0.9600,0.8154,0.6767,0.9504,0.7905,0.2627
RF-A nhị phân (baseline) | test-final,0.9414,0.9594,0.8186,0.6813,0.9491,0.7932,0.2569


In [19]:
pred_cat = mc.predict(X_fin)
print('Báo cáo đa lớp trên test-final:')
display(pd.DataFrame(classification_report(y_fin_cat, pred_cat, output_dict=True, zero_division=0)).T.round(4))

labels = sorted(set(y_fin_cat) | set(pred_cat))
cm = pd.DataFrame(confusion_matrix(y_fin_cat, pred_cat, labels=labels), index=labels, columns=labels)
print('Ma trận nhầm lẫn đa lớp (hàng = thực tế):')
display(cm)

fz = cm.loc['Fuzzers'].sort_values(ascending=False)
print('Fuzzers thật bị đoán thành:', fz[fz > 0].to_dict())
fz_mask = (yf == 1) & (cat_fin == 'Fuzzers')
print('Recall nhị phân của Fuzzers @0,5 theo điểm đa lớp: '
      f'{float((store["RF đa lớp (1−P(Normal))|fin"][fz_mask] >= 0.5).mean()):.4f}')

Báo cáo đa lớp trên test-final:


,precision,recall,f1-score,support
Analysis,0.1354,0.2207,0.1679,213.0000
Backdoor,0.1161,0.3476,0.1740,164.0000
DoS,0.5452,0.2086,0.3017,839.0000
Exploits,0.7356,0.8495,0.7885,3780.0000
Fuzzers,0.3255,0.6298,0.4292,2218.0000
Generic,0.9601,0.6056,0.7427,715.0000
Normal,0.9455,0.8118,0.8736,16017.0000
Reconnaissance,0.8214,0.7604,0.7897,1131.0000
Shellcode,0.4638,0.5189,0.4898,185.0000
Worms,0.7273,0.3636,0.4848,22.0000


Ma trận nhầm lẫn đa lớp (hàng = thực tế):


,Analysis,Backdoor,DoS,Exploits,Fuzzers,Generic,Normal,Reconnaissance,Shellcode,Worms
Analysis,47,72,18,36,26,0,0,14,0,0
Backdoor,41,57,13,14,22,0,1,14,2,0
DoS,52,104,175,370,68,2,28,34,5,1
Exploits,52,101,54,3211,127,5,109,100,19,2
Fuzzers,50,95,19,75,1397,2,547,17,16,0
Generic,0,7,12,193,43,433,18,4,5,0
Normal,84,0,10,307,2545,9,13003,3,56,0
Reconnaissance,21,55,14,133,21,0,19,860,8,0
Shellcode,0,0,5,16,40,0,27,1,96,0
Worms,0,0,1,10,3,0,0,0,0,8


Fuzzers thật bị đoán thành: {'Fuzzers': 1397, 'Normal': 547, 'Backdoor': 95, 'Exploits': 75, 'Analysis': 50, 'DoS': 19, 'Reconnaissance': 17, 'Shellcode': 16, 'Generic': 2}
Recall nhị phân của Fuzzers @0,5 theo điểm đa lớp: 0.7998


## Thực nghiệm 7 — Chọn model và ngưỡng vận hành

Hai bước tách bạch:

1. **Chọn model** bằng PR-AUC trên `test-dev` (không bao giờ dùng `test-final`).
2. **Chọn ngưỡng** cũng trên `test-dev`, theo ràng buộc FPR ≤ 10%, tối đa hoá Recall.

Sau đó `test-final` được chạm **một lần**.
Nhánh 39 feature có mặt trong bảng chỉ để đối chiếu — nó **không** đủ tư cách được chọn.

In [20]:
arms = {
    'RF-A nhị phân (baseline)': (store['RF-A nhị phân (baseline)|dev'], store['RF-A nhị phân (baseline)|fin']),
    'HGB boosting': (store['HGB boosting|dev'], store['HGB boosting|fin']),
    'RF đa lớp (1−P(Normal))': (store['RF đa lớp (1−P(Normal))|dev'], store['RF đa lớp (1−P(Normal))|fin']),
}
ref_arms = {'[ablation] RF 39 feature (ô nhiễm)': (abl_proba['B: 39 feature (+8 ô nhiễm)|test-dev'],
                                                   abl_proba['B: 39 feature (+8 ô nhiễm)|test-final'])}

pt_dev, ci_dev, draws_dev = paired_bootstrap(y_dev, {k: (v[0], 0.5) for k, v in arms.items()}, B=1000, seed=7)
print('CHỌN MODEL trên test-dev — PR-AUC kèm CI 95%:')
display(pd.DataFrame({'pr_auc_dev': pt_dev['pr_auc'],
                      'CI': [f'[{ci_dev[k]["pr_auc"][0]:.4f}, {ci_dev[k]["pr_auc"][1]:.4f}]'
                             for k in pt_dev.index]}).round(5))

base = 'RF-A nhị phân (baseline)'
for k in arms:
    if k == base:
        continue
    r = diff_ci(draws_dev, k, base).loc['pr_auc']
    print(f'  {k:28s} ΔPR-AUC={r["hiệu_TB"]:+.5f} CI[{r["CI_thấp"]:+.5f}, {r["CI_cao"]:+.5f}] '
          f'có ý nghĩa={bool(r["loại_trừ_0"])} p={r["p_hai_phía"]:.3f}')

selected = pt_dev['pr_auc'].idxmax()
print(f'\n=> Chọn theo PR-AUC cao nhất trên test-dev: {selected}')

CHỌN MODEL trên test-dev — PR-AUC kèm CI 95%:


,pr_auc_dev,CI
RF-A nhị phân (baseline),0.94291,"[0.9398, 0.9460]"
HGB boosting,0.94332,"[0.9401, 0.9463]"
RF đa lớp (1−P(Normal)),0.94532,"[0.9421, 0.9485]"


  HGB boosting                 ΔPR-AUC=+0.00039 CI[-0.00099, +0.00181] có ý nghĩa=False p=0.594
  RF đa lớp (1−P(Normal))      ΔPR-AUC=+0.00245 CI[+0.00137, +0.00348] có ý nghĩa=True p=0.000

=> Chọn theo PR-AUC cao nhất trên test-dev: RF đa lớp (1−P(Normal))


In [21]:
thresholds = {k: threshold_at_fpr(y_dev, v[0], FPR_MAX) for k, v in {**arms, **ref_arms}.items()}
print(f'Ngưỡng chọn trên test-dev với ràng buộc FPR <= {FPR_MAX}:')
for k, t in thresholds.items():
    m = metrics_at(y_dev, {**arms, **ref_arms}[k][0], t)
    print(f'  {k:38s} thr={t:.4f} | trên test-dev: recall={m["recall"]:.4f} fpr={m["fpr"]:.4f}')

Ngưỡng chọn trên test-dev với ràng buộc FPR <= 0.1:
  RF-A nhị phân (baseline)               thr=0.6949 | trên test-dev: recall=0.8578 fpr=0.1000
  HGB boosting                           thr=0.7244 | trên test-dev: recall=0.8551 fpr=0.1000
  RF đa lớp (1−P(Normal))                thr=0.6859 | trên test-dev: recall=0.8684 fpr=0.1000
  [ablation] RF 39 feature (ô nhiễm)     thr=0.7309 | trên test-dev: recall=0.8920 fpr=0.1000


In [22]:
final_rows = {}
for k, (_, p_fin) in {**arms, **ref_arms}.items():
    final_rows[f'{k} @0,50'] = metrics_at(y_fin, p_fin, 0.5)
    final_rows[f'{k} @FPR≤10% (thr={thresholds[k]:.4f})'] = metrics_at(y_fin, p_fin, thresholds[k])
final_rows['run1 @0,500 (lịch sử)'] = metrics_at(y_fin, proba_runs['run1'], thr1)
final_rows['run2 @0,515 (lịch sử)'] = metrics_at(y_fin, proba_runs['run2'], thr2)
final_rows['run3 @0,711 (lịch sử)'] = metrics_at(y_fin, proba_runs['run3'], thr3)

fin_df = pd.DataFrame(final_rows).T
print('ĐÁNH GIÁ CUỐI trên test-final (chạm một lần):')
display(fin_df[['threshold'] + cols].round(4))
fin_df[['threshold'] + cols].round(6).to_csv(os.path.join(out_dir, 'consolidated_test_final.csv'))

pt_fin, ci_fin, draws_fin = paired_bootstrap(y_fin, {k: (v[1], 0.5) for k, v in {**arms, **ref_arms}.items()},
                                             B=1000, seed=11)
print('Hiệu PR-AUC so với baseline trên test-final:')
for k in {**arms, **ref_arms}:
    if k == base:
        continue
    r = diff_ci(draws_fin, k, base).loc['pr_auc']
    print(f'  {k:38s} ΔPR-AUC={r["hiệu_TB"]:+.5f} CI[{r["CI_thấp"]:+.5f}, {r["CI_cao"]:+.5f}] '
          f'có ý nghĩa={bool(r["loại_trừ_0"])} p={r["p_hai_phía"]:.3f}')

ĐÁNH GIÁ CUỐI trên test-final (chạm một lần):


,threshold,accuracy,precision,recall,f1,fpr,roc_auc,pr_auc,TN,FP,FN,TP
"RF-A nhị phân (baseline) @0,50",0.5000,0.8186,0.6813,0.9491,0.7932,0.2569,0.9594,0.9414,11903.0,4114.0,472.0,8795.0
RF-A nhị phân (baseline) @FPR≤10% (thr=0.6949),0.6949,0.8825,0.8259,0.8607,0.8430,0.1050,0.9594,0.9414,14336.0,1681.0,1291.0,7976.0
"HGB boosting @0,50",0.5000,0.8231,0.6878,0.9473,0.7970,0.2487,0.9591,0.9417,12033.0,3984.0,488.0,8779.0
HGB boosting @FPR≤10% (thr=0.7244),0.7244,0.8787,0.8203,0.8569,0.8382,0.1086,0.9591,0.9417,14277.0,1740.0,1326.0,7941.0
"RF đa lớp (1−P(Normal)) @0,50",0.5000,0.8270,0.6945,0.9424,0.7997,0.2398,0.9607,0.9427,12176.0,3841.0,534.0,8733.0
RF đa lớp (1−P(Normal)) @FPR≤10% (thr=0.6859),0.6859,0.8846,0.8259,0.8682,0.8465,0.1059,0.9607,0.9427,14321.0,1696.0,1221.0,8046.0
"[ablation] RF 39 feature (ô nhiễm) @0,50",0.5000,0.8345,0.6969,0.9708,0.8113,0.2443,0.9690,0.9527,12104.0,3913.0,271.0,8996.0
[ablation] RF 39 feature (ô nhiễm) @FPR≤10% (thr=0.7309),0.7309,0.8951,0.8305,0.8968,0.8624,0.1059,0.9690,0.9527,14321.0,1696.0,956.0,8311.0
"run1 @0,500 (lịch sử)",0.5000,0.8156,0.6773,0.9491,0.7905,0.2616,0.9548,0.9264,11827.0,4190.0,472.0,8795.0
"run2 @0,515 (lịch sử)",0.5152,0.8287,0.6965,0.9438,0.8015,0.2379,0.9600,0.9420,12206.0,3811.0,521.0,8746.0


Hiệu PR-AUC so với baseline trên test-final:
  HGB boosting                           ΔPR-AUC=+0.00027 CI[-0.00118, +0.00162] có ý nghĩa=False p=0.688
  RF đa lớp (1−P(Normal))                ΔPR-AUC=+0.00132 CI[+0.00006, +0.00253] có ý nghĩa=True p=0.046
  [ablation] RF 39 feature (ô nhiễm)     ΔPR-AUC=+0.01124 CI[+0.00923, +0.01325] có ý nghĩa=True p=0.000


In [23]:
op = {}
for k, (_, p_fin) in {**arms, **ref_arms}.items():
    pred = p_fin >= thresholds[k]
    atk = yf == 1
    op[k] = {c: float(pred[atk & (cat_fin == c)].mean()) for c in sorted(set(cat_fin[atk]))}
print(f'Recall theo loại tấn công tại điểm vận hành FPR≤{FPR_MAX} (test-final):')
display(pd.DataFrame(op).round(4))

Recall theo loại tấn công tại điểm vận hành FPR≤0.1 (test-final):


,RF-A nhị phân (baseline),HGB boosting,RF đa lớp (1−P(Normal)),[ablation] RF 39 feature (ô nhiễm)
Analysis,0.9812,0.9765,1.0000,0.9437
Backdoor,0.9939,0.9939,0.9939,0.9878
DoS,0.9511,0.9535,0.9583,0.9845
Exploits,0.9556,0.9534,0.9616,0.9804
Fuzzers,0.5496,0.5374,0.5676,0.6249
Generic,0.9776,0.9762,0.9748,0.9902
Reconnaissance,0.9788,0.9779,0.9779,0.9920
Shellcode,0.8000,0.8054,0.8000,0.9622
Worms,0.9545,0.9545,0.9545,1.0000


## Kiểm tra độ bền theo seed

Kết luận "đa lớp tốt hơn" chỉ dựa trên một chênh lệch rất nhỏ, nên cần biết nó có lớn hơn
nhiễu của chính thuật toán hay không. Cách chia dữ liệu giữ nguyên (seed 42);
chỉ `random_state` của rừng thay đổi. Đo trên `test-dev`, không chạm `test-final`.

In [24]:
seed_rows = []
for seed in [42, 7, 2024]:
    rf_s = make_rf(random_state=seed, **best_params).fit(X_tr, y_tr)
    mc_s = make_rf(random_state=seed, **best_params).fit(X_tr, y_tr_cat)
    ni = list(mc_s.classes_).index('Normal')
    a = average_precision_score(y_dev, rf_s.predict_proba(X_dev)[:, 1])
    b = average_precision_score(y_dev, 1.0 - mc_s.predict_proba(X_dev)[:, ni])
    seed_rows.append({'seed': seed, 'nhị phân': a, 'đa lớp': b, 'hiệu': b - a})
    print(f'seed {seed}: nhị phân {a:.5f} | đa lớp {b:.5f} | hiệu {b - a:+.5f}')

sd = pd.DataFrame(seed_rows).set_index('seed')
display(sd.round(5))
print(f'Đa lớp thắng ở {int((sd["hiệu"] > 0).sum())}/{len(sd)} seed | '
      f'hiệu trung bình {sd["hiệu"].mean():+.5f} | '
      f'biên độ nhiễu của nhị phân giữa các seed {sd["nhị phân"].max() - sd["nhị phân"].min():.5f}')

seed 42: nhị phân 0.94291 | đa lớp 0.94532 | hiệu +0.00241


seed 7: nhị phân 0.94299 | đa lớp 0.94512 | hiệu +0.00213


seed 2024: nhị phân 0.94275 | đa lớp 0.94507 | hiệu +0.00232


,nhị phân,đa lớp,hiệu
seed,,,
42,0.94291,0.94532,0.00241
7,0.94299,0.94512,0.00213
2024,0.94275,0.94507,0.00232


Đa lớp thắng ở 3/3 seed | hiệu trung bình +0.00229 | biên độ nhiễu của nhị phân giữa các seed 0.00024


## Kết luận

**Cải thiện model thật sự (có kiểm định):**

- Lần 1 → lần 2 là cải thiện thật: ΔPR-AUC ≈ +0,0156, CI 95% không chứa 0.
- Đa lớp theo `attack_cat` nhỉnh hơn nhị phân một chút nhưng nhất quán: +0,0023 trên test-dev
  (3/3 seed), +0,0013 trên test-final. Hiệu ứng nhỏ, lớn hơn nhiễu seed khoảng 10 lần.

**Chỉ là đổi điểm vận hành, không phải cải thiện model:**

- Lần 2 → lần 3: ΔPR-AUC ≈ −0,0003, CI chứa 0. Cùng model lần 3 cắt ở 0,50 cho kết quả
  không phân biệt được với lần 2. Toàn bộ mức tăng Accuracy/Precision đến từ ngưỡng.

**Không có ý nghĩa thống kê:**

- HistGradientBoosting so với Random Forest: ΔPR-AUC ≈ +0,0003, CI chứa 0.
  Thuật toán không phải nút thắt.

**Vấn đề chưa giải quyết:**

- Lệch phân bố train/test vẫn còn nguyên: CV ≈ 0,983 so với test ≈ 0,941.
- Fuzzers vẫn chiếm phần lớn False Negative ở mọi cấu hình đã thử.
- 8 feature ô nhiễm đáng giá ≈ +0,011 PR-AUC, nhưng **không được** đưa trở lại model chính thức.